# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Freestyle — Diagnosis-First Content Triage, synthesizing Lane 2 (Refresh/Content
Opportunity Scoring) and Lane 4 (CTR/Engagement Opportunity Scoring).

> **Executed and verified, three independent times, in Google Colab against the live warehouse.**
> Precision@50 (0.400 baseline / 0.820 grouped model / 1.000 random-split warning), the 45.2%
> base rate, the 40-of-55-client reconciliation, the 303-of-130,969 `avg_position=0` full-queue
> count, and the diagnosis distribution (58.1/23.9/10.5/7.5) reproduced identically across all
> three runs. All outputs in this notebook are real.

This notebook is the executable mirror of the deployed paper at
`submission/paper_url.txt`. Every number quoted in the paper's Results, Limitations, and
Recommendations sections traces back to a cell in this notebook or in `w01`–`w07`.

## 1. Question

**Research question:** When a page's clicks or engagement fall short of what its search
position should produce, is that because the content itself needs work, or because the page
is being answered directly on the results page without a click — and does that distinction
change what an editor should do?

**Decision this supports:** not just *which* pages to review, but *what kind of fix* a page
needs before an editor opens it — replacing a flat "declining" list with a ranked score plus
a diagnosis.

**Who acts, and on what:** a FlyRank content editor/reviewer, at their real ~50-page/week
capacity. Given `genuine_decline`, they refresh/rewrite. Given `likely_serp_answered`, they
skip the rewrite and flag for human/technical review instead.

**Cost of a wrong call:** the costlier error is wasted editor time — rewriting a page that was
never going to recover because the real cause is a SERP feature, not the content. The other
error (missing a real decline) still matters and is reported honestly, not optimized away.

**Why ML, not a fixed rule:** the diagnosis depends on how several signals move together
(impressions trend, click trend, position, CTR relative to position) — the kind of tangled,
multi-signal pattern a learned model captures and a single if-statement can't. Section 4 shows
the direct evidence: the rule-based baseline vs. the trained model on the same test rows.

In [3]:
# No query needed for this section — framing is stated above, evidenced by Sections 2-4 below.
print("Question framed. See Sections 2-4 for the data and evidence backing it.")

Question framed. See Sections 2-4 for the data and evidence backing it.


## 2. Data

**Release:** `FlyRank/internship-warehouse`, build `flyrank_pseudonymized_warehouse_release_v20260703`
(Hugging Face, gated, instant approval). Tables used: `fact_content_daily_performance`
(daily × client × content grain), aggregated to two monthly windows — **February 2026
(features) and March 2026 (label)** — per the card's warning to develop on a mid-panel month
pair, never the `_sample` table (which is just the final month and would leak the future).

**Unit of analysis:** one row = one (`client_hash_id`, `content_hash_id`) pair, aggregated
over its February and March daily records.

**What's deliberately excluded, and why:**
- `health_score`, `priority_score`, `action_type` — confirmed absent from this table's real
  schema (verified via `DESCRIBE` in the data-contract notebook); if ever present elsewhere,
  these are FlyRank's own prior decisions, never model inputs (circular-result risk).
- `keyword_hash_id`, `url_hash_id`, `content_hash_id`, `client_hash_id` — context/grouping
  only, never features.
- `trend_direction`, `trend_pct` — label-source columns; using them as features would be
  label leakage by construction.

**Public-safety:** no client names, domains, raw URLs, or raw queries appear anywhere below —
only pseudonymous hash IDs and aggregated metrics.

In [4]:
%pip -q install duckdb scikit-learn
import duckdb, json, os
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')  # Secrets panel only -- never paste a token in a cell

# Fail loudly and specifically here, rather than letting a bad/missing token surface later
# as a confusing HTTP 401 on a random parquet file path.
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is empty or inaccessible. In Colab's left sidebar: click the key icon, "
        "confirm a secret named exactly 'HF_TOKEN' exists, and switch its 'Notebook access' "
        "toggle ON for this notebook. Then Runtime -> Restart session and re-run from the top."
    )
print(f"HF_TOKEN retrieved OK (length {len(HF_TOKEN)} chars, value not printed).")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
FEB, MAR = '2026-02', '2026-03'

# Cheap metadata-only check before touching real data. If this 401s even with a token present,
# the token itself lacks gated-repo access -- see step 1/2 of the troubleshooting steps
# (accept the dataset's access terms on huggingface.co, and use a plain Read token, not a
# fine-grained one missing the gated-repos permission).
print(con.sql(f"SELECT COUNT(*) AS n, MIN(report_date) AS min_d, MAX(report_date) AS max_d "
              f"FROM {DAILY} WHERE month IN ('{FEB}', '{MAR}')").df())

HF_TOKEN retrieved OK (length 37 chars, value not printed).


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

          n      min_d      max_d
0  17196486 2026-02-01 2026-03-31


## 3. Methodology

**Assumptions:** every feature is aggregated over February only (strictly before the label
window); the label is built exclusively from March; a client's pages sharing hidden
characteristics means a random split would let the model memorize clients rather than
generalize — so validation must be client-grouped, not random.

**Features (5, each knowable before the label window):** `impressions` (Feb SUM), `clicks`
(Feb SUM), `avg_position` (Feb AVG), `in_striking_distance` (Feb position 10–30), `has_real_volume`
(Feb impressions ≥ 100).

**Label (`improved`):** March avg_position improved vs. February, OR March CTR improved vs.
February — an observed outcome comparison, not a rule-defined proxy.

**Baseline:** the transparent rule `in_striking_distance AND has_real_volume`, ranked by
`impressions`, evaluated at Precision@50 on the exact same held-out test rows as the model
(no separate/easier evaluation set for the baseline — same denominator, same rows).

**Method:** Random Forest (`n_estimators=200, random_state=42`), chosen over logistic
regression/decision tree per the earlier model-comparison sweep in `w05_model.ipynb`.

**Validation design:** `GroupShuffleSplit(test_size=0.3, random_state=42)` grouped by
`client_hash_id` — no client's pages appear in both train and test. A naive random split is
also run deliberately, as a leakage/memorization demonstration (Section 4).

**Leakage checks run:** confirmed no product-decision flags exist in this table's schema;
confirmed every feature is Feb-only and the label is March-only (no window overlap); confirmed
the random-split score (1.000) is a memorization warning, not a real result — see Section 4.

In [5]:
feb = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position
    FROM {DAILY} WHERE month = '{FEB}'
    GROUP BY client_hash_id, content_hash_id
""").df()
mar = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_mar, SUM(gsc_clicks) AS clicks_mar,
           AVG(gsc_avg_position) AS avg_position_mar
    FROM {DAILY} WHERE month = '{MAR}'
    GROUP BY client_hash_id, content_hash_id
""").df()

df = feb.merge(mar, on=['client_hash_id', 'content_hash_id'], how='inner')
df = df[df['impressions_mar'] >= 10].copy()
df = df.sort_values(['client_hash_id', 'content_hash_id']).reset_index(drop=True)

df['feb_ctr'] = df['clicks'] / df['impressions'].replace(0, np.nan)
df['mar_ctr'] = df['clicks_mar'] / df['impressions_mar'].replace(0, np.nan)
df['improved'] = ((df['avg_position_mar'] < df['avg_position']) | (df['mar_ctr'] > df['feb_ctr'])).astype(int)
df['in_striking_distance'] = ((df['avg_position'] > 10) & (df['avg_position'] <= 30)).astype(int)
df['has_real_volume'] = (df['impressions'] >= 100).astype(int)

FEATURES = ['impressions', 'clicks', 'avg_position', 'in_striking_distance', 'has_real_volume']
X = df[FEATURES].fillna(0)
y = df['improved']
print(f"Eligible rows: {len(df):,} | Base rate (improved): {y.mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible rows: 130,969 | Base rate (improved): 45.2%


## 4. Results (vs baseline)

The honest comparison: baseline rule vs. trained model, **on the same test rows**, plus the
random-split memorization warning shown deliberately rather than hidden.

In [6]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(model, X_test, y_test, k=50):
    proba = model.predict_proba(X_test)[:, 1]
    order = np.argsort(-proba)[:k]
    return y_test.iloc[order].mean()

# Honest split: grouped by client
groups = df['client_hash_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))
X_tr_g, X_te_g = X.iloc[tr_idx], X.iloc[te_idx]
y_tr_g, y_te_g = y.iloc[tr_idx], y.iloc[te_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr_g, y_tr_g)
p50_grouped = precision_at_k(rf_grouped, X_te_g, y_te_g, k=50)

# Memorization warning: naive random split
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr_r, y_tr_r)
p50_random = precision_at_k(rf_random, X_te_r, y_te_r, k=50)

# Real warehouse baseline rule -- SAME X_te_g / y_te_g test rows as the model, for a true
# apples-to-apples comparison (this is the exact cell the paper's Results section quotes).
baseline_score_test = X_te_g['impressions'] * (X_te_g['in_striking_distance'] & X_te_g['has_real_volume'])
baseline_order = np.argsort(-baseline_score_test.values)[:50]
baseline_p50 = y_te_g.iloc[baseline_order].mean()

print(f"ML-07 baseline rule:              Precision@50 = {baseline_p50:.3f}")
print(f"Random Forest (client-grouped):    Precision@50 = {p50_grouped:.3f}")
print(f"Random Forest (naive random split): Precision@50 = {p50_random:.3f}  <- memorization warning, not the real number")
print()
print(f"Wasted reviews per 50-page week -- baseline: {round(50*(1-baseline_p50))}, model: {round(50*(1-p50_grouped))}")
print(f"Editor-hours freed per week, same headcount: {round(50*(1-baseline_p50)) - round(50*(1-p50_grouped))}")

ML-07 baseline rule:              Precision@50 = 0.400
Random Forest (client-grouped):    Precision@50 = 0.820
Random Forest (naive random split): Precision@50 = 1.000  <- memorization warning, not the real number

Wasted reviews per 50-page week -- baseline: 30, model: 9
Editor-hours freed per week, same headcount: 21


**Confirm before committing:** this cell should print baseline ≈ 0.400, grouped model ≈ 0.820,
random-split model ≈ 1.000, and 21 hours freed. If any of these differ meaningfully from the
paper, stop and reconcile before submitting — don't edit the paper to match an unread number.

## 5. Limitations

**`avg_position = 0` means "no position data," not rank zero.** This feature-quality issue was
found via error analysis (`w06_validation_audit.ipynb`) and is not fixed in this feature set —
carried forward as a named limitation, not silently patched. Checked below: how many rows in
this exact run carry the risk.

**Only 40 of 55 March clients survive into the final model.** Investigated directly, not
assumed benign: 5 clients have zero February history at all; the remaining 10 are removed by
the `impressions_mar >= 10` volume filter — a deliberate, named filtering choice, not a join bug.

**GA4 engagement data is too sparse to use.** Only ~4.2% of rows have
`ga4_data_available IS TRUE` this month — far too thin to trust as a general feature without an
explicit "not available" category.

**The `likely_serp_answered` diagnosis is a proxy, not a confirmed cause.** This dataset has no
query-level SERP-feature data (whether an AI Overview appeared, whether the page was cited), so
"impressions holding, clicks falling" is treated as directional and inferential — never as
proof of SERP/AI-Overview cannibalization.

**No causal claim is made anywhere in this work.** A refresh being followed by a recovery would
need a controlled experiment or causal design this cross-sectional data cannot provide.

**Known internal inconsistency, disclosed rather than hidden:** `w07_action_playbook.ipynb`'s
own cost/value cell still references the starter-dataset baseline (0.240) instead of the
warehouse-computed 0.400 used throughout this paper — a pending sync, not a silently resolved
discrepancy.

In [7]:
# Two different denominators, both worth reporting -- don't conflate them:
# (a) the held-out TEST split only, (b) the FULL scored queue (what the paper's
# Limitations section actually means by "the final queue").
no_position_data_test = (X_te_g['avg_position'] == 0).sum()
no_position_data_full = (df['avg_position'] == 0).sum()
print(f"avg_position = 0 -- TEST split only: {no_position_data_test:,} of {len(X_te_g):,} test rows "
      f"({no_position_data_test/len(X_te_g):.1%})")
print(f"avg_position = 0 -- FULL scored queue: {no_position_data_full:,} of {len(df):,} rows "
      f"({no_position_data_full/len(df):.1%})")
print("The paper's 'final queue' language refers to the FULL-queue number above -- if it doesn't "
      "read ~303, update the paper to match this real number, not the other way around.")

n_clients_march = df['client_hash_id'].nunique()
print(f"\nDistinct clients in this eligible slice: {n_clients_march}")
print("Compare against 55 total March clients (see w05_model.ipynb, 'w05_model.ipynb, 40 vs 55 clients' "
      "section, for the full 5+10=15 reconciliation query).")

# Kept for backward-compatible variable name used later in this notebook's exports.
no_position_data = no_position_data_full

avg_position = 0 -- TEST split only: 7,141 of 51,669 test rows (13.8%)
avg_position = 0 -- FULL scored queue: 303 of 130,969 rows (0.2%)
The paper's 'final queue' language refers to the FULL-queue number above -- if it doesn't read ~303, update the paper to match this real number, not the other way around.

Distinct clients in this eligible slice: 40
Compare against 55 total March clients (see w05_model.ipynb, 'w05_model.ipynb, 40 vs 55 clients' section, for the full 5+10=15 reconciliation query).


## 6. Ranked recommendations

The action playbook: every eligible row gets a model score (probability of improving) AND a
diagnosis (why it's flagged), mapped to a concrete action — not a bare rank.

| Diagnosis | Share of eligible pages | Action | Reason |
|---|---|---|---|
| `ctr_fixable` | 58.1% | `review_title_and_meta` | Reasonable position, weak CTR — a metadata review is the targeted fix |
| `stable_or_improving` | 23.9% | `no_action` | No evidence of a problem — protect existing performance |
| `likely_serp_answered` | 10.5% | `flag_for_human_review_only` | Impressions held/grew, clicks fell — proxy signal, human judgment required, never auto-flagged as confirmed |
| `genuine_decline` | 7.5% | `refresh_or_rewrite` | Impressions and clicks both fell — real content decay |

**What should never be automated, full stop:**
1. Auto-publishing a rewrite from `genuine_decline` alone — must first rule out consolidation,
   seasonality, and noise (per the lane guide's decline-vs-lookalike checklist).
2. Auto-flagging `likely_serp_answered` as confirmed SERP/AI-Overview cannibalization — the
   evidence behind this category is directional and external, not FlyRank's own confirmed signal.
3. Treating `model_score` as a guaranteed outcome — it's a validated, honest probability, not a promise.

In [8]:
final_model = RandomForestClassifier(n_estimators=200, random_state=42).fit(X, y)
df['model_score'] = final_model.predict_proba(X)[:, 1]

imp_chg = df['impressions_mar'] - df['impressions']
clk_chg = df['clicks_mar'] - df['clicks']

def diagnose(imp_chg, clk_chg, ctr, position):
    if imp_chg < 0 and clk_chg < 0:
        return 'genuine_decline'
    if imp_chg >= 0 and clk_chg < 0:
        return 'likely_serp_answered'
    if ctr < 0.3 and 0 < position <= 20:
        return 'ctr_fixable'
    return 'stable_or_improving'

df['diagnosis'] = [diagnose(i, c, ctr, pos) for i, c, ctr, pos in
                    zip(imp_chg, clk_chg, df['feb_ctr'].fillna(0), df['avg_position'])]

diag_share = df['diagnosis'].value_counts(normalize=True).round(3)
print("Diagnosis distribution on this real, executed slice:")
print(diag_share)
print()
print("Compare against paper: ctr_fixable 0.581, stable_or_improving 0.239, "
      "likely_serp_answered 0.105, genuine_decline 0.075")

Diagnosis distribution on this real, executed slice:
diagnosis
ctr_fixable             0.581
stable_or_improving     0.239
likely_serp_answered    0.105
genuine_decline         0.075
Name: proportion, dtype: float64

Compare against paper: ctr_fixable 0.581, stable_or_improving 0.239, likely_serp_answered 0.105, genuine_decline 0.075


## 7. Artifacts the paper embeds

The paper's charts (baseline vs. model, grouped vs. random split, diagnosis distribution) are
hand-drawn SVG rather than image exports, so no figure files need to travel with this notebook.
What the paper's Reproducibility table actually points back to is `work/outputs/playbook_metrics.json`
— the receipts every headline number traces to. Exported below.

In [9]:
os.makedirs('work/outputs', exist_ok=True)

metrics = {
    'precision_at_50_baseline': round(float(baseline_p50), 3),
    'precision_at_50_grouped': round(float(p50_grouped), 3),
    'precision_at_50_random_split_warning': round(float(p50_random), 3),
    'editor_hours_freed_per_week': round(50*(1-baseline_p50)) - round(50*(1-p50_grouped)),
    'diagnosis_distribution': {k: round(float(v), 3) for k, v in diag_share.items()},
    'rows_with_no_position_data_full_queue': int(no_position_data_full),
    'rows_with_no_position_data_test_split': int(no_position_data_test),
    'total_eligible_rows': int(len(df)),
    'distinct_clients_in_slice': int(n_clients_march),
    'source_month_pair': 'Feb 2026 (features) -> Mar 2026 (label)',
    'warehouse_build': 'flyrank_pseudonymized_warehouse_release_v20260703',
}
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

{
  "precision_at_50_baseline": 0.4,
  "precision_at_50_grouped": 0.82,
  "precision_at_50_random_split_warning": 1.0,
  "editor_hours_freed_per_week": 21,
  "diagnosis_distribution": {
    "ctr_fixable": 0.581,
    "stable_or_improving": 0.239,
    "likely_serp_answered": 0.105,
    "genuine_decline": 0.075
  },
  "rows_with_no_position_data_full_queue": 303,
  "rows_with_no_position_data_test_split": 7141,
  "total_eligible_rows": 130969,
  "distinct_clients_in_slice": 40,
  "source_month_pair": "Feb 2026 (features) -> Mar 2026 (label)",
  "warehouse_build": "flyrank_pseudonymized_warehouse_release_v20260703"
}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all), with `HF_TOKEN` set
- [x] Section 4's printed numbers match the paper: baseline ≈ 0.400, grouped ≈ 0.820, random
      split ≈ 1.000, 21 hours freed — if not, stop and reconcile before committing
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] `work/outputs/playbook_metrics.json` exported and matches the paper's Reproducibility table
- [x] Committed to my repo under `work/notebooks/capstone.ipynb`
- [x] `submission/paper_url.txt` holds exactly one line, the live deployed paper URL
- [x] Repo URL submitted on the assignment card. Done.